# Etape 2 : Questions-Réponses

## Introduction

Dans ce notebook je mets en place deux approches pour trouver les réponses les plus proches sémentiquement paralnt de la réponse initiale.

- **Word2Vec**
Word2Vec permet de creer pour chaque mot un vecteur selon le context donné par les corpus. Deux mots proches sementiquement auront donc des vecteurs "similaires". Cependant pour les réponses nous travaillons sur de phrases, j'ai donc choisis que chaque phrase serait vectoriser par un vercteur moyen des vecteurs des mots qui la compose. Cette approche me semblant peu rigoureuse il faudrait encore identifier ses limitations et les incoherences pouvant en résulter.

- **Doc2Vec**
Doc2Vec est davantage coherent puisqu'il cree directement des vecteur pour des paragraphe et non des mots. J'ai donc choisi de l'utiliser egalement afin de pouvoir comparer les deux approches.

In [162]:
#imports
import pandas as pd
from gensim.models import Word2Vec
from gensim.models import Doc2Vec
import numpy as np
import gensim.utils
import gensim.models.doc2vec

In [145]:
#data
data = pd.read_csv('../../data/train.csv') 
data = data.dropna(subset=['Context','Response'])
data_unique = data.drop_duplicates(subset=['Context','Response']) 
test_data = data_unique.sample(n=10,random_state=42) 
train_data = data_unique.drop(test_data.index) 
train_data.to_csv('../../data/train_unique_e2p2.csv',index=False) 
test_data.to_csv('../../data/test_unique_e2p2.csv',index=False)
test_data.head()



,Context,Response
3201,What makes a healthy marriage last?,This answer varies based on you relationship. ...
3365,I have a friend that who I used to be in a rel...,"It is not the case of being right or wrong, in..."
1359,My mother takes care of niece whom my sister a...,This sounds like a possible boundary issue. Bo...
1702,I am a heterosexual male in my late 20s. I fin...,If you enjoy cross-dressing and are comfortabl...
3397,Or how to send him somewhere that can help him...,Your dad needs to be aware that he has a probl...


### Fonctions utilitaires

In [146]:
def read_corpus_from_df(df, tokens_only=False):
    for i, text in enumerate(df['Response']):
        tokens = gensim.utils.simple_preprocess(str(text))
        if tokens_only:
            yield tokens
        else:
            yield gensim.models.doc2vec.TaggedDocument(tokens, [i])

In [147]:
import textwrap

def print_wrapped(label, text, width=100):
    print(f"{label}")
    print(textwrap.fill(text, width=width))
    print()

## Approche Word2Vec

### Références

- [Gensim Word2Vec documentation](https://radimrehurek.com/gensim/models/word2vec.html)


### Principe

Word2Vec permet d'entraine notre model sur les tokens du corpus.

Pour chaque phrase  **phrase** on fait alors la **moyenne** des vecteurs de ses mots afin d'obtenir son vecteur moyen.

On compare ensuite la question à toutes les réponse grace la **similarité cosinus**



### Tokenisation pour Word2Vec

In [148]:
from gensim.models import Word2Vec
import gensim.utils

train_tokens_w2v = [
    gensim.utils.simple_preprocess(str(text))
    for text in pd.concat([train_data['Response'], train_data['Context']])

]

print(f"Nombre de réponses tokenizd : {len(train_tokens_w2v)}")
print(f"Exemple de tokens : {train_tokens_w2v[0][:10]}")

Nombre de réponses tokenizd : 5476
Exemple de tokens : ['if', 'everyone', 'thinks', 'you', 're', 'worthless', 'then', 'maybe', 'you', 'need']


### Entraînement du modèle Word2Vec

- **vector_size** : dimensions du vecteur
- **min_count** : nombre d'occurence minimale à partir du quel un mot est pris en compte
- **epochs** : nombre d'itération sur le corpus
- **workers** : permet de paralleliser l'entrainement
- **window** : permet d'elargir le contexte pris en compte au tour du mot

In [149]:
w2v_model = Word2Vec(sentences=train_tokens_w2v, vector_size=80, min_count=1,epochs=80, workers=4,window=10)

#Exemple
print(f"Le mot 'depression' apparait {w2v_model.wv.get_vecattr('depression', 'count')} fois dans le corpus d'entraînement.")
print(f"Représentation sous forme de vecteur à 80 dimensions du mot 'depression':\n {w2v_model.wv['depression']}")

Le mot 'depression' apparait 650 fois dans le corpus d'entraînement.
Représentation sous forme de vecteur à 80 dimensions du mot 'depression':
 [ 3.496401    4.877616    2.2246845  -0.7615405   4.0963764  -0.1172832
  2.855038   -0.37020692  1.8282874  -1.4253206  -0.9866628  -5.4058604
 -2.243775    0.34482363  0.1449339   3.524731    4.7192545  -3.233073
  5.9639363   3.7508647  -1.6003592   0.35791096  0.12088592  0.03433361
 -3.5943105  -3.4024136  -1.6816329  -2.5697815  -2.9384425   5.8110924
  1.2746886   3.3479526  -2.0494032   7.1846266   1.6854782  -0.18210079
  2.1225097  -0.30635643 -0.73416734  0.73599255  2.9427915   0.31553861
 -0.1510534   2.4640868  -6.395138   -5.033202    0.8783119  -2.5927672
  4.072692    5.1058445  -0.576745   -0.13665685  4.5209403  -0.91872287
  0.36868274 -2.603435   -0.82531303  1.495168   -0.08559589 -1.2371744
  0.60830563  4.6531453   0.06893563  0.2861287  -0.23198235 -0.78551495
  2.9503446   1.5776641   3.980242    5.257479   -3.3618438 

### Vectorisation des réponses (moyenne des vecteurs de mots)

In [166]:
def sentence_vector(tokens, model):
    vecs = [model.wv[word] for word in tokens if word in model.wv]
    if len(vecs) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

#nettoyage des reponses
responses_raw = train_data['Response'].str.strip().reset_index(drop=True)

#vectorisation de ttes les réponses
all_vectors = np.array([
    sentence_vector(gensim.utils.simple_preprocess(str(r)), w2v_model)
    for r in responses_raw
])

#suppression des duplicats
rounded = np.round(all_vectors, decimals=5)
_, unique_indices = np.unique(rounded, axis=0, return_index=True)
unique_indices = np.sort(unique_indices)

responses_train_w2v = responses_raw.iloc[unique_indices].reset_index(drop=True)
response_vectors_w2v = all_vectors[unique_indices]

print(f"rep avant deduplicaton : {len(responses_raw)}")
print(f"rep après deduplicaton : {len(responses_train_w2v)}")
print(f"Matrice de vecteurs de responses finale : {response_vectors_w2v.shape}")

rep avant deduplicaton : 2738
rep après deduplicaton : 2019
Matrice de vecteurs de responses finale : (2019, 80)


### Fonction de recherche des K réponses les plus proches

In [167]:
from sklearn.metrics.pairwise import cosine_similarity

def k_response_w2v(question, k):
    #on vectorize la question
    q_tokens = gensim.utils.simple_preprocess(str(question))
    q_vec = sentence_vector(q_tokens, w2v_model).reshape(1, -1)

    #on mesure de la similarité avec toutes les réponse

    similarities = cosine_similarity(q_vec, response_vectors_w2v).flatten()

    #et on recup les k meilleurs indices
    top_k_indices = similarities.argsort()[-k:][::-1]

    return responses_train_w2v.iloc[top_k_indices].tolist(), similarities[top_k_indices]

### Test du modèle sur le jeu de test

In [168]:
for doc_id in range(len(test_data)):
    question = test_data.iloc[doc_id]['Context']
    true_response = test_data.iloc[doc_id]['Response']

    top_k_responses, top_k_scores = k_response_w2v(question, k=3)

    print("=== DOCUMENT DE TEST ===\n")
    print_wrapped('Context:', question)
    print_wrapped('Réponse initiale :', true_response)

    print(u'\n=== TOP 3 RÉPONSES SUGGÉRÉES PAR LE MODÈLE ===\n')
    for i, (resp, score) in enumerate(zip(top_k_responses, top_k_scores)):
        print(f"--- TOP {i+1} (score: {score:.4f}) ----")
        print_wrapped('', resp)

    print("\n" + "="*80 + "\n")

=== DOCUMENT DE TEST ===

Context:
What makes a healthy marriage last?

Réponse initiale :
This answer varies based on you relationship. However, I do believe their are some basic fundamental
areas that are beneficial for a healthy marriage:1.) Effective Communication2.) Trust3.)
Love/Passion4.) Loyalty. 5.) Unconditional Positive Regard. Everyone has their favorite qualities
they feel best fit a marriage. However, these are what I think are great starting points. 


=== TOP 3 RÉPONSES SUGGÉRÉES PAR LE MODÈLE ===

--- TOP 1 (score: 0.6143) ----

I would focus on YOU right now. We cannot control him, his actions, his love, or his decisions. But
we can work on you. Think about a few things: What do you want? What do you love about him? What
made you two separate? What do you think about being in a relationship where your partner does not
love you? Does that seem fair? He may want to work things out or he may be done. He may be done for
a short period of time or be done forever. No one ca

## Approche Doc2Vec

### Références

- [Gensim Doc2Vec example](https://radimrehurek.com/gensim/auto_examples/tutorials/run_doc2vec_lee.html)
- [Gensim Doc2Vec documentation](https://radimrehurek.com/gensim/models/doc2vec.html)


### Tokenisation pour Doc2Vec

In [153]:
train_corpus = list(read_corpus_from_df(train_data))
test_corpus  = list(read_corpus_from_df(test_data, tokens_only=True))
print(train_corpus[:2])

[TaggedDocument(words=['if', 'everyone', 'thinks', 'you', 're', 'worthless', 'then', 'maybe', 'you', 'need', 'to', 'find', 'new', 'people', 'to', 'hang', 'out', 'with', 'seriously', 'the', 'social', 'context', 'in', 'which', 'person', 'lives', 'is', 'big', 'influence', 'in', 'self', 'esteem', 'otherwise', 'you', 'can', 'go', 'round', 'and', 'round', 'trying', 'to', 'understand', 'why', 'you', 're', 'not', 'worthless', 'then', 'go', 'back', 'to', 'the', 'same', 'crowd', 'and', 'be', 'knocked', 'down', 'again', 'there', 'are', 'many', 'inspirational', 'messages', 'you', 'can', 'find', 'in', 'social', 'media', 'maybe', 'read', 'some', 'of', 'the', 'ones', 'which', 'state', 'that', 'no', 'person', 'is', 'worthless', 'and', 'that', 'everyone', 'has', 'good', 'purpose', 'to', 'their', 'life', 'also', 'since', 'our', 'culture', 'is', 'so', 'saturated', 'with', 'the', 'belief', 'that', 'if', 'someone', 'doesn', 'feel', 'good', 'about', 'themselves', 'that', 'this', 'is', 'somehow', 'terrible',

### Entrainement du model

- **vector_size** : dimensions du vecteur
- **min_count** : nombre d'occurence minimale à partir du quel un mot est pris en compte
- **epochs** : nombre d'itération sur le corpus
- **workers** : permet de paralleliser l'entrainement
- **dm** : algo utilisé (j'ai pris celui par default pour l'instant)
- **window** : permet d'elargir le contexte pris en compte au tour du mot

- **total_example** : nombre de phrases

In [154]:
model = gensim.models.doc2vec.Doc2Vec(vector_size=50, min_count=1, epochs=40, workers=4, window=5)
model.build_vocab(train_corpus)
model.train(train_corpus, total_examples=model.corpus_count, epochs=model.epochs)

In [155]:
#Exemple
print(f"Le mot 'depression' apparait {model.wv.get_vecattr('depression', 'count')} fois dans le corpus d'entraînement.")
print(f"Représentation sous forme de vecteur à 50 dimensions du mot 'depression':\n {model.wv['depression']}")

Le mot 'depression' apparait 420 fois dans le corpus d'entraînement.
Représentation sous forme de vecteur à 50 dimensions du mot 'depression':
 [ 1.210844    0.88414854  0.7796724  -0.5139064  -3.572986    0.8306611
 -3.9659626   1.7846905   0.64834523 -0.50252676  3.908725    2.0691736
 -1.1318251  -0.365168   -1.9364517  -0.2894521   0.6453412  -0.44264072
  0.6829848   1.4519982   2.3482993   1.1298999   1.4997278  -0.96810484
 -1.9508733   2.607789   -0.4746346  -1.4669969  -0.49992874 -2.3864517
  1.4600439  -2.3584049  -0.22863789 -1.5184752   0.5668561  -1.4635762
  1.6733809  -1.9091362  -0.23799573 -0.6116535   1.183276    0.7767527
  1.432475   -0.36932838 -1.2991086   0.9901586  -0.26893768  1.321888
  1.9670902  -0.25704068]


### Verification du model par inférence sur les données d'entrainement

In [156]:
ranks = []
second_ranks = []
for doc_id in range(len(train_corpus)):
    inferred_vector = model.infer_vector(train_corpus[doc_id].words)
    sims = model.dv.most_similar([inferred_vector], topn=len(model.dv))
    rank = [docid for docid, sim in sims].index(doc_id)
    ranks.append(rank)

    second_ranks.append(sims[1])

In [157]:
import collections

counter = collections.Counter(ranks)
print(counter)

Counter({0: 2028, 1: 707, 2: 2, 60: 1})


### Test du model sur le jeu de test

In [158]:
# Pick a random document from the test corpus and infer a vector from the model
for doc_id in range(len(test_corpus)):
    q_tokens = gensim.utils.simple_preprocess(str(test_data.iloc[doc_id]['Context']))
    inferred_vector = model.infer_vector(q_tokens, epochs=100)
    sims = model.dv.most_similar([inferred_vector], topn=len(model.dv))

    # Context et réponse initiale
    print("=== DOCUMENT DE TEST ===\n")
    print("Context : ", ' '.join(test_data.iloc[doc_id]['Context'].split()))
    print_wrapped('Réponse initiale :', ' '.join(test_corpus[doc_id]))

    # 3 meilleures réponses suggérées (on skip la 1ère car c'est la réponse elle-même)
    print(u'\n=== TOP 3 RÉPONSES SUGGÉRÉES PAR LE MODÈLE ===\n')
    for i in range(3):
        print(f"--- TOP {i+1} (score: {sims[i][1]:.4f}) ---")
        print_wrapped('', ' '.join(train_corpus[sims[i][0]].words))


=== DOCUMENT DE TEST ===

Context :  What makes a healthy marriage last?
Réponse initiale :
this answer varies based on you relationship however do believe their are some basic fundamental
areas that are beneficial for healthy marriage effective communication trust love passion loyalty
unconditional positive regard everyone has their favorite qualities they feel best fit marriage
however these are what think are great starting points


=== TOP 3 RÉPONSES SUGGÉRÉES PAR LE MODÈLE ===

--- TOP 1 (score: 0.6744) ---

absolutely not it is never too much the most important thing is that you are reaching out to get
help therapy helps you to develop healthier coping strategies and that can help reduce the anxiety
and depression as well as improve your sleep this can all be done at pace that is best for you your
therapist can help you process all of this in safe and supportive space

--- TOP 2 (score: 0.6614) ---

thank you for sharing your history you do not have too many issues to address in 

## Approche BERT

*******\*TODO\********


## Conclusion

In [159]:
from sklearn.metrics.pairwise import cosine_similarity

# similirité moey w2v
w2v_scores = []
for text in test_data['Context']:
    q_tokens = gensim.utils.simple_preprocess(str(text))
    q_vec = sentence_vector(q_tokens, w2v_model).reshape(1, -1)
    # On prend le score du Top 1
    sims = cosine_similarity(q_vec, response_vectors_w2v).flatten()
    w2v_scores.append(np.max(sims))

#sims moey d2v
d2v_scores = []
for text in test_data['Context']:
    q_tokens = gensim.utils.simple_preprocess(str(text))
    inferred_vector = model.infer_vector(q_tokens)
    #score le + proche
    sims = model.dv.most_similar([inferred_vector], topn=1)
    d2v_scores.append(sims[0][1])

#tableau recapitulatif
comparison_results = pd.DataFrame({
    'Métrique': ['Moyenne Top 1', 'Score Max', 'Score Min'],
    'Word2Vec': [np.mean(w2v_scores), np.max(w2v_scores), np.min(w2v_scores)],
    'Doc2Vec': [np.mean(d2v_scores), np.max(d2v_scores), np.min(d2v_scores)]
})

print("Synthèse automatique des performances (Similarité Cosinus) :")
comparison_results.set_index('Métrique')

Synthèse automatique des performances (Similarité Cosinus) :


,Word2Vec,Doc2Vec
Métrique,,
Moyenne Top 1,0.829471,0.608988
Score Max,0.958706,0.706653
Score Min,0.614329,0.524266


In [160]:
# Extraction des moyennes pour la décision
avg_w2v = comparison_results.loc[0, 'Word2Vec']
avg_d2v = comparison_results.loc[0, 'Doc2Vec']
winner = "Word2Vec" if avg_w2v > avg_d2v else "Doc2Vec"

print(f"=== ANALYSE DE COHÉRENCE ===")
print(f"Model Word2Vec (vector_size={w2v_model.vector_size}, epochs={w2v_model.epochs}) : {avg_w2v:.4f}")
print(f"Model Doc2Vec (vector_size={model.vector_size}, epochs={model.epochs}) : {avg_d2v:.4f}")
print(f"\nConclusion : Le model **{winner}** est  le plus performant.")


=== ANALYSE DE COHÉRENCE ===
Model Word2Vec (vector_size=80, epochs=80) : 0.8295
Model Doc2Vec (vector_size=50, epochs=40) : 0.6090

Conclusion : Le model **Word2Vec** est  le plus performant.


### Word2Vec

Les resultats de Word2Vec sont assez surprenemment coherents... 

**TODO : meilleur interpretation des resultats**

### Doc2Vec

resultats peu insatisfaisants, performe mal sur les petit textes

**TODO : meilleur interpretation des resultats**
